# TASK 3 · Fraud Detection

## Credit Card Fraud Detection with SMOTE

**Objective:** Build an end-to-end machine learning pipeline to detect fraudulent financial transactions from a heavily imbalanced dataset.

**Tech Stack:** Python, pandas, scikit-learn, imbalanced-learn (`SMOTE`), matplotlib, seaborn, Jupyter Notebook.

### Dataset
This notebook uses the Kaggle **Credit Card Fraud Detection** benchmark dataset containing **284,807 transactions**, of which **492 are fraudulent**.

The target column is `Class`:
- `0` = legitimate transaction
- `1` = fraudulent transaction

> **Important:** SMOTE is applied **only to the training data**, after the stratified train/test split. This prevents synthetic samples from leaking information into the test set.


## 1. Install / Import Libraries

If your environment does not already contain the required packages, run the installation cell below once.

In [ ]:
# Run this only if required:
# %pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn

In [ ]:
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    accuracy_score
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 2. Load the Dataset

Download `creditcard.csv` from Kaggle and place it in the same folder as this notebook.

The Kaggle dataset is approximately 151 MB, so the CSV itself is not bundled with this notebook.

In [ ]:
# The loader checks common locations used by Jupyter, VS Code, Google Colab, and Kaggle.

possible_paths = [
    "creditcard.csv",
    "./creditcard.csv",
    "/content/creditcard.csv",
    "/kaggle/input/creditcardfraud/creditcard.csv",
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)

if data_path is None:
    raise FileNotFoundError(
        "creditcard.csv was not found. Download it from Kaggle and place it "
        "in the same folder as this notebook."
    )

df = pd.read_csv(data_path)

print(f"Loaded from: {data_path}")
print(f"Dataset shape: {df.shape}")
display(df.head())

## 3. Basic Data Inspection

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe().T)

## 4. Class Imbalance Analysis

This dataset is **extremely imbalanced**. Fraud represents only about **0.172%** of all transactions.

A model that predicts every transaction as legitimate could obtain roughly **99.83% accuracy while detecting zero fraud cases**. Therefore, accuracy alone is not a useful measure of fraud-detection quality.

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_percent = df["Class"].value_counts(normalize=True).sort_index() * 100

imbalance_table = pd.DataFrame({
    "Transaction Type": ["Legitimate", "Fraud"],
    "Count": [class_counts.get(0, 0), class_counts.get(1, 0)],
    "Percentage": [class_percent.get(0, 0), class_percent.get(1, 0)]
})

display(imbalance_table)

fraud_percentage = class_percent.get(1, 0)
print(f"Fraudulent transactions: {fraud_percentage:.3f}%")

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Class")
plt.title("Class Distribution")
plt.xlabel("Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.show()

## 5. EDA — Transaction Amount Distribution

The `Amount` variable is highly skewed. Fraudulent and legitimate transactions can have very different amount distributions, so both the raw and log-scaled views are useful.

`log1p(Amount)` is used only for visualization here; the modeling pipeline keeps the original feature and scales the numerical variables before Logistic Regression.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df,
    x="Amount",
    hue="Class",
    bins=100,
    stat="density",
    common_norm=False,
    element="step"
)
plt.xlim(0, df["Amount"].quantile(0.995))
plt.title("Transaction Amount Distribution — Fraud vs Legitimate")
plt.xlabel("Transaction Amount")
plt.ylabel("Density")
plt.show()

In [ ]:
df["Amount_log"] = np.log1p(df["Amount"])

plt.figure(figsize=(10, 5))
sns.histplot(
    data=df,
    x="Amount_log",
    hue="Class",
    bins=80,
    stat="density",
    common_norm=False,
    element="step"
)
plt.title("Log-Scaled Transaction Amount Distribution")
plt.xlabel("log1p(Amount)")
plt.ylabel("Density")
plt.show()

## 6. EDA — Time-of-Day Analysis

The benchmark dataset's `Time` column is the number of seconds elapsed since the first transaction. It is **not a real clock timestamp**.

For exploratory analysis, we convert elapsed seconds into a 24-hour cycle using modulo 86,400. This gives a relative hour-of-day representation and is useful for observing temporal patterns, but it should not be interpreted as the exact real-world clock time.

In [ ]:
SECONDS_PER_DAY = 24 * 60 * 60

df["Hour"] = ((df["Time"] % SECONDS_PER_DAY) // 3600).astype(int)

hourly = (
    df.groupby(["Hour", "Class"])
      .size()
      .reset_index(name="Transactions")
)

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=hourly,
    x="Hour",
    y="Transactions",
    hue="Class",
    marker="o"
)
plt.title("Relative Time-of-Day Transaction Pattern")
plt.xlabel("Relative Hour of Day")
plt.ylabel("Number of Transactions")
plt.xticks(range(0, 24))
plt.show()

In [ ]:
hourly_rate = (
    df.groupby("Hour")["Class"]
      .mean()
      .mul(100)
      .reset_index(name="Fraud Rate (%)")
)

plt.figure(figsize=(12, 5))
sns.barplot(data=hourly_rate, x="Hour", y="Fraud Rate (%)")
plt.title("Fraud Rate by Relative Hour")
plt.xlabel("Relative Hour")
plt.ylabel("Fraud Rate (%)")
plt.xticks(range(0, 24))
plt.show()

## 7. Why Accuracy Is Misleading for Fraud Detection

**Accuracy = (correct predictions) / (all predictions).**

With a fraud rate of only about 0.172%, legitimate transactions dominate the dataset. A classifier can therefore achieve extremely high accuracy by predicting almost everything as legitimate.

For fraud detection, the more important questions are:

- **Recall:** Of all actual fraud cases, how many did we catch?
- **Precision:** Of the transactions flagged as fraud, how many really were fraud?
- **F1-score:** A balance between precision and recall.
- **ROC-AUC:** Measures how well the model ranks fraud above legitimate transactions across classification thresholds.

In a fraud-prevention system, **Recall is often especially important** because a false negative means an actual fraudulent transaction was missed. However, extremely high recall can produce many false positives, so precision and the operational cost of investigating alerts must also be considered.

## 8. Train/Test Split with Stratification

We separate the target (`Class`) from the predictors and use `stratify=y`.

This guarantees that both training and test sets preserve approximately the original fraud/non-fraud ratio.

**SMOTE must not be applied before this split**, because doing so can cause data leakage.

In [ ]:
# Drop helper EDA columns; keep the original modeling variables.
model_df = df.drop(columns=["Amount_log", "Hour"], errors="ignore").copy()

X = model_df.drop(columns=["Class"])
y = model_df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).mul(100).round(3))

print("\nTest class distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).mul(100).round(3))

## 9. Feature Scaling

`Amount` and `Time` are on different scales from the PCA-transformed `V1`–`V28` variables.

We standardize all features before Logistic Regression. Random Forest does not require scaling, but using the same transformed feature matrix keeps the experiment straightforward.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed without using information from the test set.")

## 10. Apply SMOTE to the Training Data

**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic minority-class samples using the neighborhood of existing minority examples.

We apply it **only to `X_train` and `y_train`**. The test set remains untouched and keeps the original real-world class distribution.

In [ ]:
print("Before SMOTE:", Counter(y_train))

smote = SMOTE(random_state=RANDOM_STATE)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("After SMOTE:", Counter(y_train_smote))

## 11. Model 1 — Logistic Regression

Logistic Regression is a strong and interpretable baseline for binary classification.

Because the data is heavily imbalanced, we train it on the SMOTE-balanced training set and evaluate it on the **original imbalanced test set**.

In [ ]:
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

log_reg.fit(X_train_smote, y_train_smote)

lr_pred = log_reg.predict(X_test_scaled)
lr_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression trained.")

## 12. Model 2 — Random Forest

Random Forest is an ensemble of decision trees and can capture nonlinear relationships that Logistic Regression may miss.

For a first practical run, we use 100 trees. On a larger production dataset, the number of trees and other hyperparameters should be tuned against validation performance and latency requirements.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]

print("Random Forest trained.")

## 13. Evaluation — Precision, Recall, F1 and ROC-AUC

We evaluate on the untouched test set.

- **Precision** penalizes false alarms.
- **Recall** penalizes missed fraud.
- **F1-score** balances precision and recall.
- **ROC-AUC** summarizes ranking performance across thresholds.
- Accuracy is shown only for context, not as the main decision metric.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression + SMOTE", y_test, lr_pred, lr_prob),
    evaluate_model("Random Forest + SMOTE", y_test, rf_pred, rf_prob)
])

display(results.round(4))

In [ ]:
print("=== Logistic Regression ===")
print(classification_report(y_test, lr_pred, digits=4, zero_division=0))

print("=== Random Forest ===")
print(classification_report(y_test, rf_pred, digits=4, zero_division=0))

## 14. Confusion Matrices

A confusion matrix makes the fraud trade-off concrete:

- **True Negative (TN):** legitimate transaction correctly accepted
- **False Positive (FP):** legitimate transaction incorrectly flagged
- **False Negative (FN):** fraud incorrectly missed
- **True Positive (TP):** fraud correctly detected

For fraud detection, **FN is usually the most costly error**, because it represents a fraudulent transaction that escaped detection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name, pred in zip(
    axes,
    ["Logistic Regression", "Random Forest"],
    [lr_pred, rf_pred]
):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(f"{name} — Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 15. ROC-AUC Curves

The ROC curve shows the trade-off between:

- **True Positive Rate (Recall)**
- **False Positive Rate**

The AUC summarizes the model's ability to rank fraudulent transactions above legitimate ones.

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

plt.figure(figsize=(8, 6))
plt.plot(
    lr_fpr, lr_tpr,
    label=f"Logistic Regression (AUC = {roc_auc_score(y_test, lr_prob):.4f})"
)
plt.plot(
    rf_fpr, rf_tpr,
    label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_prob):.4f})"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")

plt.title("ROC-AUC Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

## 16. Which Metric Matters Most?

There is no single universal threshold, because the correct operating point depends on the financial and customer-service cost of errors.

### Recall vs Precision

**High Recall**
- Catches more actual fraud.
- Reduces missed fraudulent transactions.
- Can increase false positives and customer friction.

**High Precision**
- Makes fewer false fraud alerts.
- Reduces unnecessary transaction declines or manual investigations.
- Can allow more fraud to pass through.

For many fraud-screening systems, **Recall is the first priority**, because missing fraud can directly cause financial loss. However, the production system should choose a probability threshold using the business cost of false negatives versus false positives.

Therefore, this notebook reports **Precision, Recall, F1-score and ROC-AUC together** instead of selecting a model from accuracy alone.

## 17. Feature Importance — Logistic Regression Coefficients

For Logistic Regression, the magnitude of a coefficient indicates how strongly a feature influences the prediction after scaling.

Positive coefficients push predictions toward fraud; negative coefficients push them toward legitimate transactions.

In [ ]:
feature_names = X.columns

lr_coef = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": log_reg.coef_[0],
    "Absolute Coefficient": np.abs(log_reg.coef_[0])
}).sort_values("Absolute Coefficient", ascending=False)

display(lr_coef.head(15))

plt.figure(figsize=(10, 6))
top_lr = lr_coef.head(15).sort_values("Coefficient")
sns.barplot(data=top_lr, x="Coefficient", y="Feature")
plt.title("Top Logistic Regression Coefficients")
plt.show()

## 18. Feature Importance — Random Forest

Random Forest provides impurity-based feature importance scores. These indicate which variables contributed most to the ensemble's decisions.

Feature importance is useful for interpretation, but it should not automatically be treated as proof of causality.

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

display(rf_importance.head(15))

plt.figure(figsize=(10, 6))
top_rf = rf_importance.head(15).sort_values("Importance")
sns.barplot(data=top_rf, x="Importance", y="Feature")
plt.title("Top Random Forest Feature Importances")
plt.show()

## 19. Scalability — 1 Million Transactions per Hour

A production system receiving **1,000,000 transactions/hour** would need approximately **278 transactions/second on average**.

A scalable architecture could use:

1. **Streaming ingestion:** Kafka, Kinesis, Pub/Sub, or another event-streaming platform.
2. **Feature service:** Compute transaction features consistently in real time.
3. **Low-latency model serving:** Deploy the trained model behind a lightweight API or model-serving service.
4. **Horizontal scaling:** Run multiple model-serving instances behind a load balancer.
5. **Batch/stream monitoring:** Track fraud rate, recall, precision, latency, and data drift continuously.
6. **Asynchronous investigation:** Send high-risk transactions to a review queue instead of blocking every transaction.
7. **Periodic retraining:** Fraud patterns change over time, so the model should be retrained using recent labeled transactions.
8. **Threshold management:** Use different risk thresholds depending on the cost of false positives and false negatives.

### Practical model choice

For very high throughput, Logistic Regression can be attractive because inference is lightweight and fast. Random Forest can still work at this scale, but model size, number of trees, feature-generation time, and serving latency must be benchmarked.

SMOTE is primarily a **training-time** technique. We do not run SMOTE on live transactions; production inference uses the trained model directly.

## 20. Final Conclusion

This project demonstrates an end-to-end fraud-detection workflow:

- Loaded and inspected a highly imbalanced financial transaction dataset.
- Quantified the fraud percentage.
- Explored transaction amount and relative time-of-day patterns.
- Explained why accuracy is misleading for rare-event classification.
- Used a stratified train/test split.
- Applied **SMOTE only to the training data**.
- Trained **Logistic Regression** and **Random Forest** models.
- Evaluated with **Precision, Recall, F1-score and ROC-AUC**.
- Examined confusion matrices and ROC curves.
- Investigated feature importance and model coefficients.
- Discussed how the solution could be scaled to approximately **1 million transactions per hour**.

### Key takeaway

For fraud detection, a model is not successful simply because it has high accuracy. The goal is to **catch as much real fraud as possible while keeping false alerts manageable**, so Recall, Precision, F1-score and threshold-dependent business costs should guide model selection.

## References

1. **Kaggle — Credit Card Fraud Detection:** https://www.kaggle.com/mlg-ulb/creditcardfraud
2. **imbalanced-learn — SMOTE documentation:** https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html
3. **YouTube — How to handle imbalanced datasets in Machine Learning (Python):** https://www.youtube.com/watch?v=flhjn6e6wnY
4. Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). *SMOTE: Synthetic Minority Over-sampling Technique*. Journal of Artificial Intelligence Research.
